# LeetCode 2500 — Delete Greatest Value in Each Row bằng Python

Link đề: https://leetcode.com/problems/delete-greatest-value-in-each-row/ · Độ khó: **Easy** · Chủ đề: Array, Matrix, Sorting, Heap

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def deleteGreatestValue(self, grid: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu quy trình lặp "xóa phần tử lớn nhất của **mỗi hàng**, lấy **lớn nhất trong các giá trị vừa xóa** cộng vào đáp án".
2. Nhận ra bài này thực chất là: **sắp xếp từng hàng**, rồi cộng **max của từng cột**.
3. Biết dùng `list.remove`, `max`, `sorted`, `zip(*matrix)` (chuyển vị ma trận) và `heapq` cho đống lớn nhất.
4. Viết 4 cách: mô phỏng đúng đề, sắp xếp hàng + max cột, `heapq` max-heap cho từng hàng, và đếm tần suất theo giá trị (`1..100`); đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: cộng mọi giá trị đã xóa thay vì chỉ cộng max mỗi vòng, sắp xếp tăng rồi lấy cột nhưng nhầm hướng, sửa trực tiếp `grid` của đầu vào, và quên rằng các hàng có cùng số cột.

## 2. Đề bài & ví dụ

Cho ma trận `grid` kích thước `m x n` gồm các số nguyên dương. Thực hiện thao tác sau **cho đến khi `grid` rỗng**:

1. Từ **mỗi hàng**, xóa phần tử có giá trị **lớn nhất** (nếu có nhiều phần tử bằng nhau, xóa một phần tử bất kỳ trong số đó).
2. Lấy **giá trị lớn nhất** trong các phần tử vừa xóa (mỗi hàng một phần tử) và **cộng vào đáp án**.

Sau mỗi thao tác, số cột của ma trận giảm đi 1. Trả về đáp án sau khi `grid` rỗng.

| Ví dụ | grid | Output | Giải thích |
|---|---|---|---|
| 1 | `[[1,2,4],[3,3,1]]` | `8` | vòng 1 xóa `4` và `3`, cộng `4`; vòng 2 xóa `2` và `3`, cộng `3`; vòng 3 xóa `1` và `1`, cộng `1`; tổng `8` |
| 2 | `[[10]]` | `10` | một vòng, xóa `10` |

Constraints: `m == grid.length`; `n == grid[i].length`; `1 <= m, n <= 50`; `1 <= grid[i][j] <= 100`.

In [1]:
grid = [[1, 2, 4], [3, 3, 1]]
ans = 0
g = [row[:] for row in grid]            # sao chép để không làm hỏng đầu vào
vong = 0
while g[0]:
    vong += 1
    xoa = []
    for row in g:
        m = max(row)
        row.remove(m)                   # xóa một phần tử lớn nhất
        xoa.append(m)
    print(f"vòng {vong}: đã xóa {xoa}, cộng {max(xoa)}, grid còn {g}")
    ans += max(xoa)
print("đáp án =", ans, "(mong đợi 8)")

vòng 1: đã xóa [4, 3], cộng 4, grid còn [[1, 2], [3, 1]]
vòng 2: đã xóa [2, 3], cộng 3, grid còn [[1], [1]]
vòng 3: đã xóa [1, 1], cộng 1, grid còn [[], []]
đáp án = 8 (mong đợi 8)


## 3. Kiến thức nền

### 3.1. `max`, `list.remove`

`max(row)` trả về phần tử lớn nhất. `row.remove(x)` xóa **lần xuất hiện đầu tiên** của `x` (lỗi `ValueError` nếu không có). Khi có nhiều phần tử bằng nhau, xóa cái nào cũng cho cùng một list kết quả vì chúng giống hệt nhau.

In [2]:
row = [3, 3, 1]
m = max(row)
row.remove(m)
print(m, row)

3 [3, 1]


### 3.2. `sorted` và `list.sort`

`sorted(row)` trả về list mới đã sắp xếp tăng dần (không đổi `row`). `row.sort()` sắp xếp **tại chỗ** (đổi chính `row`). `sorted(row, reverse=True)` cho thứ tự giảm dần.

In [3]:
row = [3, 1, 2]
print(sorted(row), row)
row.sort(reverse=True)
print(row)

[1, 2, 3] [3, 1, 2]
[3, 2, 1]


### 3.3. `zip(*matrix)` — chuyển vị ma trận (lấy từng cột)

Dấu `*` "mở gói" list các hàng thành nhiều đối số cho `zip`; `zip` ghép phần tử cùng vị trí, nên kết quả là các **cột**. Mỗi cột là một `tuple`.

In [4]:
matrix = [[1, 2, 4], [3, 3, 1]]
cols = list(zip(*matrix))
print(cols)                        # [(1, 3), (2, 3), (4, 1)]
print([max(c) for c in cols])      # max của từng cột

[(1, 3), (2, 3), (4, 1)]
[3, 3, 4]


### 3.4. `heapq` với max-heap bằng cách đổi dấu

`heapq` chỉ có **min-heap**: `heappop` lấy phần tử **nhỏ nhất**. Để lấy lớn nhất, đẩy `-x` vào heap rồi đổi dấu khi lấy ra. `heapify(list)` biến list thành heap trong `O(n)`; mỗi `heappop` tốn `O(log n)`.

In [5]:
import heapq
row = [1, 2, 4]
h = [-x for x in row]
heapq.heapify(h)
print([-heapq.heappop(h) for _ in range(len(row))])   # 4 2 1: giảm dần

[4, 2, 1]


### 3.5. Mảng đếm tần suất (counting) theo giá trị

Vì mỗi giá trị chỉ từ `1` đến `100`, ta có thể dùng mảng `cnt[0..100]` để đếm số lần xuất hiện, rồi duyệt giá trị từ lớn xuống nhỏ để sinh phần tử theo thứ tự giảm dần mà không cần `sort`.

In [6]:
row = [3, 3, 1, 100]
cnt = [0] * 101
for x in row:
    cnt[x] += 1
giam_dan = [v for v in range(100, 0, -1) for _ in range(cnt[v])]
print(giam_dan)

[100, 3, 3, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay `grid = [[1,2,4],[3,3,1]]`:

- Vòng 1: lớn nhất mỗi hàng là `4` và `3`; lấy `max = 4`.
- Vòng 2: hàng 1 còn `[1,2]` nên xóa `2`; hàng 2 còn `[3,1]` nên xóa `3`; lấy `3`.
- Vòng 3: hàng 1 còn `[1]`, hàng 2 còn `[1]`; lấy `1`.

Quan sát then chốt: ở vòng thứ `j`, mỗi hàng xóa **phần tử lớn thứ `j`** của chính hàng đó (lớn nhất, lớn nhì, lớn ba, ...). Thứ tự xóa trong mỗi hàng **không phụ thuộc vào các hàng khác**.

Nếu sắp xếp mỗi hàng giảm dần, hàng `i` trở thành `[lớn nhất, lớn nhì, ...]`, và vòng `j` chính là **cột `j`** của ma trận đã sắp xếp; đáp án vòng `j` là max của cột `j`.

In [7]:
grid = [[1, 2, 4], [3, 3, 1]]
sorted_rows = [sorted(r, reverse=True) for r in grid]
print(sorted_rows)
print("max từng cột:", [max(c) for c in zip(*sorted_rows)])
print("tổng:", sum(max(c) for c in zip(*sorted_rows)))

[[4, 2, 1], [3, 3, 1]]
max từng cột: [4, 3, 1]
tổng: 8


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng đề

**Ý tưởng:** làm theo đúng từng bước: lặp `n` vòng; mỗi vòng, với mỗi hàng tìm `max`, `remove` nó khỏi hàng, ghi nhận; cộng `max` của các giá trị đã xóa. Ta làm việc trên **bản sao** của `grid`.

In [8]:
from typing import List

class SolutionBrute:
    def deleteGreatestValue(self, grid: List[List[int]]) -> int:
        g = [row[:] for row in grid]               # sao chép, không phá đầu vào
        n = len(g[0])
        ans = 0
        for _ in range(n):
            deleted = []
            for row in g:
                m = max(row)
                row.remove(m)
                deleted.append(m)
            ans += max(deleted)
        return ans

In [9]:
sol = SolutionBrute()
print(sol.deleteGreatestValue([[1, 2, 4], [3, 3, 1]]))   # 8
print(sol.deleteGreatestValue([[10]]))                   # 10

8
10


**Độ phức tạp:** Time `O(m * n^2)`: `n` vòng, mỗi vòng `m` hàng, mỗi hàng `max` và `remove` đều `O(n)`. Space `O(m * n)` cho bản sao. **Nút thắt:** tìm lại max và xóa phần tử trong list cho mỗi vòng quá tốn; sắp xếp một lần là đủ.

## 6. Ý tưởng thuật toán / bước đột phá

Theo mục 4: vòng `j` xóa phần tử lớn thứ `j` của **mỗi** hàng, độc lập giữa các hàng. Do đó:

1. Sắp xếp **từng hàng** một lần (tăng hoặc giảm đều được, miễn nhất quán).
2. Cột thứ `j` của ma trận sắp xếp chứa các giá trị bị xóa ở vòng `j`; lấy **max cột** rồi cộng lại.

Nếu sắp xếp tăng dần thì cột `j` (từ trái) là phần tử nhỏ thứ `j`, và vòng đầu tiên xóa phần tử **cuối** của hàng. Tổng các max cột không phụ thuộc thứ tự cột vì ta chỉ cộng tất cả, nên sắp tăng hay giảm đều cho cùng tổng.

**Quy tắc:** *sort mỗi hàng, đáp án = tổng max theo cột.*

**Invariant:** sau khi sắp xếp, phần tử ở cột `j` của hàng `i` là đúng giá trị mà hàng `i` xóa ở vòng thứ `j` (đếm theo thứ tự xóa).

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp hàng rồi lấy max theo cột

Một dòng: `sum(map(max, zip(*(sorted(row) for row in grid))))`. Viết ra từng bước cho dễ hiểu.

In [10]:
from typing import List

class SolutionSort:
    def deleteGreatestValue(self, grid: List[List[int]]) -> int:
        sorted_rows = [sorted(row) for row in grid]      # sắp xếp mỗi hàng (tăng dần)
        ans = 0
        for col in zip(*sorted_rows):                    # từng cột
            ans += max(col)
        return ans

In [11]:
sol = SolutionSort()
print(sol.deleteGreatestValue([[1, 2, 4], [3, 3, 1]]))   # 8
print(sol.deleteGreatestValue([[10]]))                   # 10

8
10


**Độ phức tạp:** Time `O(m * n log n)` (sắp xếp `m` hàng độ dài `n`) cộng `O(m * n)` duyệt cột; Space `O(m * n)` cho các hàng đã sắp xếp.

### Cách 3 — `heapq`: mỗi hàng là một max-heap

Mỗi hàng chuyển thành heap các số đổi dấu. Mỗi vòng, `heappop` một lần từ mỗi heap. Không cần sắp xếp toàn bộ, nhưng cũng tốn `O(log n)` mỗi lần lấy.

In [12]:
import heapq
from typing import List

class SolutionHeap:
    def deleteGreatestValue(self, grid: List[List[int]]) -> int:
        heaps = []
        for row in grid:
            h = [-x for x in row]                        # đổi dấu để dùng min-heap thành max-heap
            heapq.heapify(h)
            heaps.append(h)
        n = len(grid[0])
        ans = 0
        for _ in range(n):
            biggest = 0
            for h in heaps:
                biggest = max(biggest, -heapq.heappop(h))
            ans += biggest
        return ans

In [13]:
sol = SolutionHeap()
print(sol.deleteGreatestValue([[1, 2, 4], [3, 3, 1]]))   # 8
print(sol.deleteGreatestValue([[10]]))                   # 10

8
10


**Độ phức tạp:** Time `O(m * n log n)` (`heapify` `O(n)` mỗi hàng, `m * n` lần `heappop` mỗi lần `O(log n)`), Space `O(m * n)`.

### Cách 4 — Đếm tần suất theo giá trị (counting sort)

Giá trị chỉ từ `1` đến `100`. Với mỗi hàng, đếm tần suất rồi sinh dãy giảm dần mà không cần `sort`. Ta cộng vào `best[j]` giá trị lớn nhất cho vòng `j`.

In [14]:
from typing import List

class SolutionCount:
    def deleteGreatestValue(self, grid: List[List[int]]) -> int:
        n = len(grid[0])
        best = [0] * n                                   # best[j] = max các giá trị xóa ở vòng j
        for row in grid:
            cnt = [0] * 101
            for x in row:
                cnt[x] += 1
            j = 0
            for v in range(100, 0, -1):                  # duyệt giá trị từ lớn xuống nhỏ
                for _ in range(cnt[v]):
                    if v > best[j]:
                        best[j] = v
                    j += 1
        return sum(best)

In [15]:
sol = SolutionCount()
print(sol.deleteGreatestValue([[1, 2, 4], [3, 3, 1]]))   # 8
print(sol.deleteGreatestValue([[10]]))                   # 10

8
10


**Độ phức tạp:** Time `O(m * (n + V))` với `V = 100` là miền giá trị (đếm `O(n)`, duyệt miền `O(V)`), Space `O(V + n)`. Hiệu quả khi `n` lớn còn `V` nhỏ.

## 8. TRACE từng bước

Trace Cách 2 với `grid = [[1,2,4],[3,3,1]]`:

| Bước | Trạng thái |
|---|---|
| sắp xếp hàng | `[[1,2,4],[1,3,3]]` |
| cột 0 | `(1, 1)` nên `max = 1`, `ans = 1` |
| cột 1 | `(2, 3)` nên `max = 3`, `ans = 4` |
| cột 2 | `(4, 3)` nên `max = 4`, `ans = 8` |

Kết quả `8`: khớp mô phỏng (thứ tự cộng khác nhưng tổng giống nhau vì ta cộng cả `n` giá trị max).

In [16]:
def trace_sort(grid):
    sorted_rows = [sorted(row) for row in grid]
    print("sau khi sắp xếp hàng:", sorted_rows)
    ans = 0
    for j, col in enumerate(zip(*sorted_rows)):
        ans += max(col)
        print(f"cột {j}: {col} -> max = {max(col)}, ans = {ans}")
    return ans

print(trace_sort([[1, 2, 4], [3, 3, 1]]))
print(trace_sort([[10]]))
print(trace_sort([[5, 5], [1, 9], [4, 2]]))

sau khi sắp xếp hàng: [[1, 2, 4], [1, 3, 3]]
cột 0: (1, 1) -> max = 1, ans = 1
cột 1: (2, 3) -> max = 3, ans = 4
cột 2: (4, 3) -> max = 4, ans = 8
8
sau khi sắp xếp hàng: [[10]]
cột 0: (10,) -> max = 10, ans = 10
10
sau khi sắp xếp hàng: [[5, 5], [1, 9], [2, 4]]
cột 0: (5, 1, 2) -> max = 5, ans = 5
cột 1: (5, 9, 4) -> max = 9, ans = 14
14


## 9. Code nộp LeetCode

Cách 2 (sắp xếp hàng + max theo cột):

In [17]:
from typing import List

class Solution:
    def deleteGreatestValue(self, grid: List[List[int]]) -> int:
        for row in grid:
            row.sort()
        return sum(max(col) for col in zip(*grid))

In [18]:
print(Solution().deleteGreatestValue([[1, 2, 4], [3, 3, 1]]))
print(Solution().deleteGreatestValue([[10]]))

8
10


**Ghi chú:** `row.sort()` sắp xếp tại chỗ (đổi `grid` của người gọi); LeetCode không yêu cầu giữ nguyên đầu vào nên chấp nhận. Trong bộ test của notebook ta luôn truyền **bản sao** để so sánh công bằng giữa các cách.

## 10. Vì sao thuật toán đúng

1. **Hàng độc lập.** Việc xóa phần tử lớn nhất của một hàng chỉ phụ thuộc vào chính hàng đó; hàng khác không ảnh hưởng. Vì vậy ở vòng `j`, hàng `i` xóa phần tử lớn thứ `j` của hàng `i` (tính theo thứ tự giảm dần).
2. **Sắp xếp biểu diễn thứ tự xóa.** Nếu sắp xếp hàng tăng dần, vòng `j` xóa phần tử thứ `j` từ **cuối**; nếu giảm dần là từ **đầu**. Phần tử bằng nhau có thể xóa theo thứ tự nào cũng cho cùng một dãy giá trị.
3. **Cột = một vòng.** Gom các phần tử cùng "hạng" (lớn thứ `j`) của mọi hàng thành một nhóm, và nhóm đó chính là một **cột** ở ma trận đã sắp xếp theo hướng tương ứng. Vòng `j` cộng `max` của nhóm đó.
4. **Tổng không đổi khi đảo thứ tự cột.** Khi sắp xếp tăng dần, cột `j` ứng với vòng `n - 1 - j`. Vì ta chỉ cộng **tất cả** `n` giá trị max nên thứ tự cộng không ảnh hưởng đáp án.
5. **Mọi hàng cùng số cột.** `n` giống nhau với mọi hàng nên cột nào cũng đủ `m` phần tử, `zip(*...)` không bị cắt ngắn.
6. **Kết thúc.** Sau đúng `n` vòng mọi hàng rỗng, ta đã cộng đủ `n` giá trị.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(m * n^2)` | `O(m * n)` (bản sao) |
| 2. Sort hàng + max cột | `O(m * n log n)` | `O(m * n)` (hoặc `O(1)` thêm nếu sort tại chỗ) |
| 3. `heapq` | `O(m * n log n)` | `O(m * n)` |
| 4. Đếm tần suất | `O(m * (n + V))`, `V = 100` | `O(V + n)` |

`m` là số hàng, `n` là số cột (`<= 50`), `V = 100` là miền giá trị. Với `m, n <= 50` tất cả đều chạy tức thì.

## 12. Bộ test

### 12.1. Ví dụ của đề, edge case và phản ví dụ

Gồm: ma trận `1 x 1`, một hàng, một cột, mọi giá trị bằng nhau, giá trị biên `1` và `100`, và ma trận mà tổng "max mỗi hàng" khác tổng đúng (để phát hiện lỗi hiểu đề). Mỗi lần gọi truyền một **bản sao** của `grid`.

In [19]:
import copy
sols = [SolutionBrute(), SolutionSort(), SolutionHeap(), SolutionCount(), Solution()]
cases = [
    ([[1, 2, 4], [3, 3, 1]], 8),                 # ví dụ 1
    ([[10]], 10),                                # ví dụ 2
    ([[5, 1, 3]], 9),                            # một hàng: mọi giá trị đều được cộng (5 + 3 + 1)
    ([[1], [7], [4]], 7),                        # một cột: chỉ một vòng, lấy max
    ([[2, 2], [2, 2]], 4),                       # toàn bằng nhau: 2 + 2
    ([[1, 1, 1], [1, 1, 1]], 3),                 # biên nhỏ nhất
    ([[100, 100], [100, 100]], 200),             # biên lớn nhất
    ([[5, 5], [1, 9], [4, 2]], 14),              # vòng 1: max(5,9,4)=9; vòng 2: max(5,1,2)=5
    ([[9, 1], [1, 9]], 10),                      # sort: [1,9],[1,9] -> max cột 1 và 9
    ([[3, 8, 1, 6], [7, 2, 9, 4]], 22),          # sort: [1,3,6,8], [2,4,7,9] -> 2 + 4 + 7 + 9
]
for grid, want in cases:
    for s in sols:
        got = s.deleteGreatestValue(copy.deepcopy(grid))
        assert got == want, (type(s).__name__, grid, got, want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 10


Kiểm tra tay vài ca: `[[5,5],[1,9],[4,2]]`: sort tăng `[5,5],[1,9],[2,4]`; cột 0 `(5,1,2)` max `5`; cột 1 `(5,9,4)` max `9`; tổng `14`. `[[3,8,1,6],[7,2,9,4]]`: sort `[1,3,6,8],[2,4,7,9]`; max cột `2, 4, 7, 9`; tổng `22`. `[[9,1],[1,9]]`: sort `[1,9],[1,9]`; max cột `1, 9`; tổng `10`.

### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh `grid` đúng constraints: `1 <= m, n <= 50`, `1 <= grid[i][j] <= 100`. Trộn kích thước nhỏ (để có nhiều giá trị trùng) và kích thước lớn.

In [20]:
import random
random.seed(2500)

def gen(max_m, max_n, max_v):
    m = random.randint(1, max_m)
    n = random.randint(1, max_n)
    return [[random.randint(1, max_v) for _ in range(n)] for _ in range(m)]

for t in range(1500):
    if t % 3 == 0:
        grid = gen(4, 4, 5)                  # nhiều giá trị trùng
    elif t % 3 == 1:
        grid = gen(10, 10, 100)
    else:
        grid = gen(50, 50, 100)              # kích thước tối đa
    m, n = len(grid), len(grid[0])
    assert 1 <= m <= 50 and 1 <= n <= 50
    assert all(len(r) == n and all(1 <= v <= 100 for v in r) for r in grid)
    ans = [s.deleteGreatestValue(copy.deepcopy(grid)) for s in sols]
    assert len(set(ans)) == 1, (grid, ans)
print("1500 test ngẫu nhiên: 5 cách khớp nhau")

1500 test ngẫu nhiên: 5 cách khớp nhau


### 12.3. Tính chất của đáp án

Với mọi `grid` hợp lệ: `max(grid) <= đáp án <= n * max(grid)` (vòng đầu đã cộng đúng `max(grid)`, mỗi vòng cộng không quá `max(grid)`); đáp án không đổi khi hoán vị các hàng hoặc hoán vị các phần tử trong cùng một hàng.

In [21]:
random.seed(5)
for _ in range(800):
    grid = gen(8, 8, 30)
    base = Solution().deleteGreatestValue(copy.deepcopy(grid))
    n = len(grid[0])
    big = max(max(r) for r in grid)
    assert big <= base <= n * big
    g2 = copy.deepcopy(grid)
    random.shuffle(g2)                                   # hoán vị hàng
    for r in g2:
        random.shuffle(r)                                # hoán vị trong hàng
    assert Solution().deleteGreatestValue(g2) == base
print("các tính chất chặn và bất biến theo hoán vị đều đúng")

các tính chất chặn và bất biến theo hoán vị đều đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cộng **mọi** giá trị đã xóa thay vì chỉ cộng max mỗi vòng

In [22]:
def sai_cong_het(grid):
    return sum(sum(row) for row in grid)                 # SAI: cộng toàn bộ phần tử

grid = [[1, 2, 4], [3, 3, 1]]
print("đúng:", Solution().deleteGreatestValue(copy.deepcopy(grid)), "| sai:", sai_cong_het(grid))
assert Solution().deleteGreatestValue(copy.deepcopy(grid)) != sai_cong_het(grid)

đúng: 8 | sai: 14


**Vì sao sai:** mỗi vòng chỉ cộng **một** giá trị (max của các giá trị xóa), không phải tất cả. Tổng toàn bộ là `14`, còn đáp án là `8`.

### Lỗi 2: Chỉ lấy max của hàng, quên sắp xếp để ghép đúng hạng

In [23]:
def sai_max_hang(grid):
    return sum(max(row) for row in grid)                 # SAI: chỉ tính vòng đầu tiên rồi cộng các hàng

grid = [[1, 2, 4], [3, 3, 1]]
print("đúng:", Solution().deleteGreatestValue(copy.deepcopy(grid)), "| sai:", sai_max_hang(grid))
assert Solution().deleteGreatestValue(copy.deepcopy(grid)) != sai_max_hang(grid)

đúng: 8 | sai: 7


**Vì sao sai:** `sum(max(row))` cộng max của **mỗi hàng** ở vòng đầu (`4 + 3 = 7`) thay vì max **giữa các hàng** (`4`) rồi mới sang vòng sau. Bài yêu cầu `n` vòng, mỗi vòng chỉ một số.

### Lỗi 3: Sắp xếp hàng theo hai chiều khác nhau

In [24]:
def sai_huong_sap_xep(grid):
    rows = [sorted(r) if i % 2 == 0 else sorted(r, reverse=True) for i, r in enumerate(grid)]   # SAI: không nhất quán
    return sum(max(c) for c in zip(*rows))

grid = [[1, 2, 4], [3, 3, 1]]
print("đúng:", Solution().deleteGreatestValue(copy.deepcopy(grid)), "| sai:", sai_huong_sap_xep(grid))
assert Solution().deleteGreatestValue(copy.deepcopy(grid)) != sai_huong_sap_xep(grid)

đúng: 8 | sai: 10


**Vì sao sai:** mọi hàng phải sắp xếp **cùng chiều** để cột `j` chứa các phần tử cùng hạng. Trộn chiều làm ghép nhầm hạng giữa các hàng.

### Lỗi 4: Lấy `min` của cột thay vì `max`

In [25]:
def sai_min_cot(grid):
    rows = [sorted(r) for r in grid]
    return sum(min(c) for c in zip(*rows))               # SAI: đề yêu cầu lớn nhất

grid = [[1, 2, 4], [3, 3, 1]]
print("đúng:", Solution().deleteGreatestValue(copy.deepcopy(grid)), "| sai:", sai_min_cot(grid))
assert Solution().deleteGreatestValue(copy.deepcopy(grid)) != sai_min_cot(grid)

đúng: 8 | sai: 6


**Vì sao sai:** đề lấy **giá trị lớn nhất** trong các giá trị bị xóa ở mỗi vòng, nên phải dùng `max` theo cột.

### Lỗi 5: Làm hỏng đầu vào khi mô phỏng

In [26]:
def sai_pha_dau_vao(grid):
    ans = 0
    for _ in range(len(grid[0])):
        deleted = []
        for row in grid:                                 # SAI: xóa trực tiếp trên grid của người gọi
            m = max(row)
            row.remove(m)
            deleted.append(m)
        ans += max(deleted)
    return ans

grid = [[1, 2, 4], [3, 3, 1]]
before = copy.deepcopy(grid)
res = sai_pha_dau_vao(grid)
print("kết quả:", res, "| grid sau khi chạy:", grid, "(đã bị làm rỗng)")
assert res == 8 and grid != before

kết quả: 8 | grid sau khi chạy: [[], []] (đã bị làm rỗng)


**Cách tránh:** sao chép `[row[:] for row in grid]` trước khi sửa, hoặc dùng cách sắp xếp không phá cấu trúc theo hàng khi cần giữ đầu vào.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "mỗi vòng xóa lớn nhất mỗi hàng" ⇒ nghĩ ngay **sắp xếp từng hàng**: các hàng độc lập với nhau.
- Thao tác lặp lại theo cột của ma trận đã sắp xếp ⇒ **`zip(*matrix)`** lấy cột.
- Một dòng nộp bài: `return sum(max(c) for c in zip(*(sorted(r) for r in grid)))`.
- Dùng `heapq` khi không muốn sắp xếp toàn bộ hoặc khi dữ liệu đến dần.
- Miền giá trị nhỏ (`<= 100`) ⇒ có thể **đếm tần suất** thay cho sắp xếp.
- Luôn kiểm tra với ví dụ `[[1,2,4],[3,3,1]]` (đáp án `8`).

In [27]:
# Một dòng nộp bài
def one_line(grid):
    return sum(max(c) for c in zip(*(sorted(r) for r in grid)))

assert one_line([[1, 2, 4], [3, 3, 1]]) == 8
assert one_line([[10]]) == 10
print("phiên bản một dòng đúng")

phiên bản một dòng đúng


## 15. Biến thể của bài

1. **Xóa phần tử nhỏ nhất mỗi hàng, cộng min trong các giá trị xóa.** Đối xứng: sort giảm dần và lấy min theo cột.
2. **Cộng tổng các giá trị xóa (không chỉ max):** khi đó đáp án đơn giản bằng tổng toàn bộ ma trận.
3. **Các hàng có số phần tử khác nhau:** vòng `j` chỉ xét hàng còn phần tử; cần `itertools.zip_longest`.
4. **Xóa phần tử lớn nhất của cả ma trận mỗi vòng** (không theo hàng): dùng một heap toàn cục.
5. **Trả về dãy các max theo từng vòng** thay vì tổng.
6. **Phiên bản trực tuyến:** các hàng đến lần lượt; duy trì `best[j]` như Cách 4 để cập nhật.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | `max` + `remove` từng vòng | `O(m * n^2)` | `O(m * n)` | hiểu đề, làm oracle test |
| 2. Sort hàng + max cột | mỗi cột là một vòng | `O(m * n log n)` | `O(m * n)` | cách nộp bài chuẩn |
| 3. `heapq` | max-heap mỗi hàng | `O(m * n log n)` | `O(m * n)` | dữ liệu đến dần, chỉ cần lấy từng phần tử lớn nhất |
| 4. Đếm tần suất | counting theo giá trị `1..100` | `O(m * (n + V))` | `O(V + n)` | miền giá trị nhỏ, `n` lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().deleteGreatestValue([[1, 1], [2, 2]])` bằng bao nhiêu?

**Bài 2 (tính tay).** Với `grid = [[4, 1, 7], [2, 9, 3]]`, hãy liệt kê giá trị xóa từng vòng và tính đáp án.

**Bài 3 (điền TODO).** Hoàn thành `round_maxima(grid)` trả về **danh sách** các max của từng vòng (theo thứ tự vòng 1, 2, ...).

In [28]:
def round_maxima(grid):
    # TODO: sắp xếp mỗi hàng GIẢM dần, rồi với mỗi cột lấy max
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `delete_smallest_value(grid)`: mỗi vòng xóa phần tử **nhỏ nhất** của mỗi hàng và cộng **giá trị nhỏ nhất** trong số vừa xóa.

**Bài 5 (biến thể / mini challenge).** Viết `delete_greatest_ragged(rows)` cho các hàng có độ dài **khác nhau** (hàng ngắn hết thì không tham gia nữa); mỗi vòng cộng max của các giá trị xóa trong vòng đó, lặp đến khi mọi hàng rỗng. Kiểm tra với `[[3, 1], [5]]` ra `5 + 1 = 6`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: vòng 1 xóa `1` và `2`, cộng `2`; vòng 2 xóa `1` và `2`, cộng `2`; tổng `4`.
- **Mức 2 (vừa):** Bài 2: sort giảm: `[7,4,1]`, `[9,3,2]`; max cột `9, 4, 2`; tổng `15`. Bài 3: `[max(c) for c in zip(*[sorted(r, reverse=True) for r in grid])]`.
- **Mức 3 (sâu):** Bài 4: sort tăng, lấy `min` theo cột. Bài 5: sắp xếp mỗi hàng giảm dần, dùng `itertools.zip_longest(*rows, fillvalue=0)` để cột ngắn được lấp bằng `0` (giá trị dương nên không ảnh hưởng `max`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
from itertools import zip_longest

# Bài 1
assert Solution().deleteGreatestValue([[1, 1], [2, 2]]) == 4

# Bài 2
assert Solution().deleteGreatestValue([[4, 1, 7], [2, 9, 3]]) == 15

# Bài 3
def round_maxima(grid):
    rows = [sorted(r, reverse=True) for r in grid]
    return [max(c) for c in zip(*rows)]

# Bài 4
def delete_smallest_value(grid):
    rows = [sorted(r) for r in grid]                     # tăng dần: cột j là phần tử nhỏ thứ j
    return sum(min(c) for c in zip(*rows))

# Bài 5
def delete_greatest_ragged(rows):
    rs = [sorted(r, reverse=True) for r in rows]
    return sum(max(c) for c in zip_longest(*rs, fillvalue=0))

assert round_maxima([[1, 2, 4], [3, 3, 1]]) == [4, 3, 1]
assert sum(round_maxima([[4, 1, 7], [2, 9, 3]])) == 15
assert round_maxima([[10]]) == [10]
assert delete_smallest_value([[1, 2, 4], [3, 3, 1]]) == 1 + 2 + 3
assert delete_smallest_value([[5]]) == 5
assert delete_greatest_ragged([[3, 1], [5]]) == 6
assert delete_greatest_ragged([[2, 2], [2, 2]]) == Solution().deleteGreatestValue([[2, 2], [2, 2]])
random.seed(19)
for _ in range(200):
    g = gen(6, 6, 20)
    assert sum(round_maxima(g)) == Solution().deleteGreatestValue(copy.deepcopy(g))
    assert delete_greatest_ragged(g) == Solution().deleteGreatestValue(copy.deepcopy(g))
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (các hàng độc lập: sắp xếp từng hàng rồi xử lý theo cột)

```python
class Solution:
    def deleteGreatestValue(self, grid: List[List[int]]) -> int:
        for row in grid:
            row.sort()
        return sum(max(col) for col in zip(*grid))
```

**Checklist**

- [ ] Đã hiểu: mỗi vòng chỉ cộng **một** số (max của các giá trị xóa), không cộng cả hàng?
- [ ] Mọi hàng được sắp xếp **cùng chiều** trước khi lấy cột?
- [ ] Dùng `max` theo cột (không phải `min`) và cộng đủ `n` cột?
- [ ] Đã thử `[[1,2,4],[3,3,1]] -> 8` và `[[10]] -> 10`?
- [ ] Biết rằng `row.sort()` đổi đầu vào; khi cần giữ nguyên thì dùng `sorted(row)`?
- [ ] Trường hợp `1 x 1`, một hàng, một cột đều đúng?
- [ ] Signature đúng `deleteGreatestValue(self, grid: List[List[int]]) -> int` và có `from typing import List`?